# Tutorial on the usage of RSA tool

This tutorial illustrates how to use the highly-optimized Ring Stacking Analysis (RSA) tool for polymer systems.

Before starting any analysis, load the necessary modules for this class and ensure your output directories exist.

In [ ]:
import os
import numpy as np
import pandas as pd
import MDAnalysis as mda

from pysoftk.pol_analysis.tools.utils_mda import MDA_input
from pysoftk.pol_analysis.tools.utils_tools import *
from pysoftk.pol_analysis.make_micelle_whole import micelle_whole
from pysoftk.pol_analysis.ring_ring import RSA

os.makedirs('data', exist_ok=True)
network_pdb_dir = 'data/network_pdbs'
os.makedirs(network_pdb_dir, exist_ok=True)

1. Select your trajectory files. It is recommended to use a `.tpr` file for the topology and an `.xtc` file for the trajectory. Note that any MDAnalysis supported file can be used here.

In [ ]:
topology = 'data/f8bt_slab_quench.tpr'
trajectory = 'data/1_frame_traj.xtc'

The simulation where we are going to perform the analysis is on a very large dense polymer slab. Due to the high atom count, we will perform the analysis on a single frame for this tutorial.

![Image Alt Text](data/pictures_tutorial/ring_system.png)

This simulation box is filled with F8BT conjugated polymers:

![Image Alt Text](data/pictures_tutorial/ring_1_polymer.png)

This class requires minimal user input. Only the angle cutoff, distance cutoff, start, stop, and step frames are needed, as well as the name of the output file.

In [ ]:
# Name output file
results_file = 'data/rsa.parquet'

# Angle cutoff - maximum angle between plane normal vectors
ang_c = 25

# Distance cutoff - maximum distance between rings to be considered stacked (Angstroms)
dist_c = 4.5

# Start, stop, and step frame
start, stop, step = 0, 1, 1

Now, let's run the RSA stacking analysis! This version is highly optimized to calculate Singular Value Decompositions (SVD) on large systems.

In [ ]:
rsa_calc = RSA(topology, trajectory)
print("Ring Stacking analysis has started...")

# write_pdb=False is used because we will selectively generate network PDBs later
rsa_calc.stacking_analysis(dist_c, ang_c, start, stop, step, results_file, write_pdb=False)
print("Stacking analysis complete.")

## 2. Exploring the Results
The output is stored in a Pandas DataFrame and saved as a Parquet file. Because this version is optimized for large polymers, it tracks connectivity via `pol_resid`.

In [ ]:
df = pd.read_parquet(results_file)

# CLEANUP: Drop the 'atom_index' column since the optimized tool uses 'pol_resid' exclusively
if 'atom_index' in df.columns:
    df = df.drop(columns=['atom_index'])

print("Cleaned DataFrame Head:")
print(df.head())

print("\nExample of interacting polymer pairs in the first recorded stack:")
print(df['pol_resid'].iloc[0])

You can selectively generate PDB snapshots of these specific pairs to visually verify the stacking geometries inside PyMOL or VMD.

![Image Alt Text](data/pictures_tutorial/ring_stacking_snapshot_1.png)
![Image Alt Text](data/pictures_tutorial/ring_stacking_snapshot_2.png)

## 3. Network Analysis & PDB Generation
One of the most powerful features of the RSA tool is extracting the **Connected Network** of polymers. By analyzing these continuous clusters, you can determine if your system has reached percolation or remains as isolated aggregates.

In [ ]:
# Extract the networks from the results file
sev_ring = rsa_calc.find_several_rings_stacked(results_file)

if sev_ring and len(sev_ring[0]) > 0:
    clusters = sev_ring[0]
    print(f"Total isolated pi-stacked clusters found: {len(clusters)}")
    
    # Sort clusters by size to easily find the largest
    clusters_sorted = sorted(clusters, key=len, reverse=True)
    largest_cluster = clusters_sorted[0]
    
    print(f"Largest connected network contains {len(largest_cluster)} polymers.")
    
    # Convert the set to a sorted list to slice safely
    sorted_members = sorted(list(largest_cluster))
    print(f"First 20 members of largest cluster: {sorted_members[:20]}")
else:
    print("No stacked networks found with these parameters.")

The code block below demonstrates how to automatically isolate these networks and save them as independent `.pdb` files for structural analysis.

In [ ]:
# Grab the universe from our RSA calculator and set it to the first frame
u_rsa = rsa_calc.get_mda_universe()
u_rsa.trajectory[0]

if sev_ring and len(sev_ring[0]) > 0:
    # Iterate over the clusters (sorted by size)
    for i, network in enumerate(clusters_sorted, start=1):
        
        # Convert the set of resids into an MDAnalysis selection string
        resid_str = " ".join(map(str, network))
        cluster_selection = u_rsa.select_atoms(f'resid {resid_str}')
        
        # Define output path and write PDB
        pdb_filename = os.path.join(network_pdb_dir, f"polymer_network_rank_{i}.pdb")
        with mda.Writer(pdb_filename, cluster_selection.n_atoms) as W:
            W.write(cluster_selection)
            
        print(f"Saved Network {i} ({len(network)} polymers) to: {pdb_filename}")
        
        # Stop after printing the top 5 largest networks to save time
        if i == 5:
            break

You can now load `polymer_network_rank_1.pdb` into your visualization software to study the morphology of your largest connected stacking network!

![Image Alt Text](data/pictures_tutorial/network_rings_stacked.png)